# Preliminary Target Selection

This notebook aims at selecting a preliminary sample that we consider KL measurement. Note that a KL sample is first a WL sample. The specific target selection criteria are
- Size cut: spatial resolution power $R$>0.4
- Not in blending system
- More?

And the spatial resolution factor is 
$$R = (1+\mathrm{EE50}_\mathrm{PSF}^2/r_\mathrm{eff,\,gal}^2)^{-1}, $$
where $\mathrm{EE50}$ is the 50 per cent encircled energy radius and $r_\mathrm{eff,\,gal}$ is the galaxy effective radius, aka half-light radius. 



In [2]:
import numpy as np
from astropy.table import Table
from astropy.io import fits
import matplotlib.pyplot as plt

In [5]:
v1_cats_fname = [
    "/xdisk/timeifler/jiachuanxu/jwst/fengwu_catalog/v1_gds_fresco_line_list_low-ground-z_Pa_Br_jades_dr5.fits",
    "/xdisk/timeifler/jiachuanxu/jwst/fengwu_catalog/v1_gdn_fresco_line_list_low-ground-z_Pa_Br_jades_dr5.fits",
    "/xdisk/timeifler/jiachuanxu/jwst/fengwu_catalog/v1_gdn_congress_line_list_low-ground-z_Pa_Br_jades_dr5.fits"
]
GRISM_Cats = [
    Table().read(fn) for fn in v1_cats_fname
]

GRISM_names = ["FRESCO_GDS", "FRESCO_GDN", "CONGRESS_GDN"]
GRISM_bands = ["F444W", "F444W", "F444W"]
Broadband_PSF_FWHM = [0.14, 0.14, 0.14] # arcsec

In [12]:
# Count galaxies by emission line per catalog
lines = [('PaA', 'Paschen alpha'), ('PaB', 'Paschen beta'), ('BrA', 'Brackett alpha'), ('BrB', 'Brackett beta')]

header = f"{'Line':<22}" + "".join(f"{n:>15}" for n in GRISM_names)
print(header)
print("-" * (22 + 15 * 3))
for code, label in lines:
    counts = [np.sum(cat['name_line_exp'] == code) for cat in GRISM_Cats]
    print(f"{label + ' (' + code + ')':<22}" + "".join(f"{c:>15}" for c in counts))
print("-" * (22 + 15 * 3))
totals = [len(cat) for cat in GRISM_Cats]
print(f"{'Total (all lines)':<22}" + "".join(f"{t:>15}" for t in totals))

Line                       FRESCO_GDS     FRESCO_GDN   CONGRESS_GDN
-------------------------------------------------------------------
Paschen alpha (PaA)               183            109            246
Paschen beta (PaB)                 31             48             38
Brackett alpha (BrA)               12              9              0
Brackett beta (BrB)                45             68             49
-------------------------------------------------------------------
Total (all lines)                 281            251            336


In [9]:
np.sum(GRISM_Cats[0]["name_line_exp"] == "PaA")

183

In [ ]:
def get_spatial_resolution_factor(cat, band, psf_fwhm):
    # calculate the spatial resolution factor
    # R = 
    # where FWHM_galaxy is calculated as 2*sqrt(A*B) where A and B are the major and minor axis of the galaxy
    A = cat["A"] # arcsec
    B = cat["B"] # arcsec
    FWHM_galaxy = 2 * np.sqrt(A * B) # arcsec
    R = psf_fwhm / FWHM_galaxy
    return R